# P1 · PPI Network Analysis
**Input:** `data/processed/dea_results.csv`  
**Outputs:** `results/tables/hub_genes.csv` · `results/tables/ppi_edges_cytoscape.csv`
· `results/figures/ppi_network.png`

**Run order:** 01 → 02 → 03 → 04 → **P1** → P2 → P3 → P4


In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "paths.py").exists():
            return p
    raise FileNotFoundError("paths.py not found — run: python scripts/data_download.py")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paths import REPO_ROOT, PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
sys.path.insert(0, str(REPO_ROOT / "scripts"))
print(f"Repo root : {REPO_ROOT}")

In [ ]:
from ppi_functions import load_dea, query_string, build_and_score, export_ppi
from utils import plot_ppi_network
import matplotlib.pyplot as plt
print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
STRING_SCORE  = 400   # 400=medium, 700=high confidence
LOG2FC_THRESH = 1.0
PADJ_THRESH   = 0.05
TOP_VIZ_NODES = 80    # nodes shown in the network figure

## 1 · Load DEA results

In [ ]:
sig, gene_list = load_dea(PROC_DIR / "dea_results.csv",
                          log2fc_thresh=LOG2FC_THRESH,
                          padj_thresh=PADJ_THRESH)
sig.head()

## 2 · Query STRING API

In [ ]:
edges_df = query_string(gene_list,
                        string_score=STRING_SCORE,
                        batch_size=500)
edges_df.head()

## 3 · Build graph & compute hub scores

In [ ]:
G, hub_df = build_and_score(sig, edges_df)
hub_df[["gene","degree","hub_score","regulation"]].head(10)

## 4 · Visualise network

In [ ]:
fig, _ = plot_ppi_network(G, hub_df,
                          top_nodes=TOP_VIZ_NODES,
                          string_score=STRING_SCORE,
                          log2fc_thresh=LOG2FC_THRESH,
                          padj_thresh=PADJ_THRESH)
fig.savefig(FIGURES_DIR / "ppi_network.png", dpi=200, bbox_inches="tight")
plt.show()

## 5 · Export

In [ ]:
export_ppi(hub_df, G, edges_df, TABLES_DIR)